# POC — Short-horizon tree models

This notebook is the first new comparative experiment after the reconciled 5/10/15-day
regression baseline.

It answers one question:

Can the current feature set identify a small, repeatable set of stocks with unusually favorable
outcomes over the next 1–5 sessions?

The experiment compares:

- horizons of 1, 2, 3, 4, and 5 observed sessions;

- raw adjusted-close return and split-adjusted price change measured in ATR units;

- Ridge, XGBRegressor , and XGBRanker ;

- aggregate regression/ranking metrics, fixed top-k selections, complete top-score buckets,
no-trade dates, and validation subperiod stability.

The locked test period is deliberately not read. Expensive bootstrap and permutation tests are
opt-in and run only for shortlisted validation configurations.

## 1. Configuration

In [ ]:
from datetime import date
from pathlib import Path

PROVIDER = "yfinance"

# Example smoke test: ("AAK.ST", "SAAB-B.ST", "VOLV-B.ST")
TICKERS: tuple[str, ...] | None = None

DATA_CUTOFF = date(2026, 7, 24)

TRAIN_START = date(2000, 1, 1)
TRAIN_END = date(2022, 12, 31)
VALIDATION_START = date(2023, 1, 1)
VALIDATION_END = date(2024, 12, 31)
TEST_START = date(2025, 1, 1)
TEST_END = DATA_CUTOFF

HORIZONS = (1, 2, 3, 4, 5)
TARGET_FAMILIES = ("raw_return", "atr_units")
MODEL_NAMES = ("ridge", "xgboost_regressor", "xgboost_ranker")
TOP_K_VALUES = (1, 3, 5)

ATR_LENGTH = 14
MIN_ATR_FRACTION_OF_PRICE = 1e-4
MAX_ABS_ATR_UNITS = 50.0

# Relevance grades 0–4. The highest grade represents approximately the top 3% of each date.
RANK_RELEVANCE_QUANTILES = (0.50, 0.75, 0.90, 0.97)

RANDOM_SEED = 42
XGB_N_JOBS = -1
STORE_FITTED_MODELS = False
MAX_SHORTLIST_CONFIGURATIONS = 5

# Resampling is intentionally opt-in. Use small counts while iterating, then increase finalists.
RUN_RESAMPLING = False
BOOTSTRAP_ITERATIONS = 500
PERMUTATION_ITERATIONS = 250
RESAMPLING_CONFIGURATION_LIMIT = 3

XGB_REGRESSOR_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 3,
    "gamma": 0.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "n_jobs": XGB_N_JOBS,
    "random_state": RANDOM_SEED,
}

XGB_RANKER_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 3,
    "gamma": 0.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "objective": "rank:ndcg",
    "eval_metric": "ndcg@5",
    "tree_method": "hist",
    "n_jobs": XGB_N_JOBS,
    "random_state": RANDOM_SEED,
}


### Runtime policy

The 30 model runs are executed sequentially with a progress bar. Both XGBoost estimators use
multiple CPU threads internally, so outer multiprocessing would normally oversubscribe the
machine rather than accelerate this matrix. Resampling is consolidated across top-k values and
is disabled until the validation shortlist exists.

## 2. Repository, database, and canonical baseline bundle

In [ ]:
import importlib.metadata

import numpy as np
import pandas as pd
from sqlalchemy import text

from swingtrader.data.bronze.loaders import load_bronze_daily_prices
from swingtrader.data.db import resolve_database_engine
from swingtrader.data.features import DEFAULT_FEATURE_SET
from swingtrader.indicators import atr
from swingtrader.modeling.datasets import (
    UniverseSpec,
    FORWARD_RETURN_PRIMARY_TASK,
    FORWARD_RETURN_TARGET_SET,
    build_temporal_dataset,
)
from swingtrader.modeling.datasets.labels import add_forward_return_targets
from swingtrader.modeling.experiments import (
    ExperimentSpec,
    FixedTemporalSplitter,
    ModelSpec,
    TemporalSplitSpec,
)
from swingtrader.modeling.training import LOGISTIC_REGRESSION_MODEL_TYPE


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current directory.")


repo_root = find_repo_root(Path.cwd())
database_path = repo_root / "data" / "swingtrader.sqlite"
database_url = f"sqlite+pysqlite:///{database_path.as_posix()}"
engine = resolve_database_engine(database_url=database_url)

if TICKERS is None:
    with engine.connect() as connection:
        ticker_rows = connection.execute(
            text(
                """
                SELECT DISTINCT ticker
                FROM bronze_market_daily_prices
                WHERE provider = :provider
                ORDER BY ticker
                """
            ),
            {"provider": PROVIDER},
        ).fetchall()
    resolved_tickers = tuple(row[0] for row in ticker_rows)
else:
    resolved_tickers = tuple(TICKERS)

if not resolved_tickers:
    raise RuntimeError(
        f"No bronze tickers were found for provider {PROVIDER!r}. "
        "Populate the local database or set TICKERS explicitly."
    )

package_versions = {}
for package in ("numpy", "pandas", "scikit-learn", "xgboost"):
    try:
        package_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        package_versions[package] = "not installed"

{
    "database_path": database_path,
    "ticker_count": len(resolved_tickers),
    "first_tickers": resolved_tickers[:10],
    "package_versions": package_versions,
}


In [ ]:
feature_set = DEFAULT_FEATURE_SET

placeholder_model = ModelSpec(
    name="short_horizon_tree_model_poc",
    version="1",
    model_type=LOGISTIC_REGRESSION_MODEL_TYPE,
    hyperparameters={},
    feature_columns=None,
)

universe = UniverseSpec(
    name="local_bronze_short_horizon_universe",
    version="1",
    provider=PROVIDER,
    tickers=resolved_tickers,
)

split_spec = TemporalSplitSpec(
    name="short_horizon_fixed_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

experiment_spec = ExperimentSpec(
    name="short_horizon_tree_models",
    version="1",
    feature_set=feature_set,
    target_set=FORWARD_RETURN_TARGET_SET,
    task=FORWARD_RETURN_PRIMARY_TASK,
    universe=universe,
    data_cutoff=DATA_CUTOFF,
    split=split_spec,
    model=placeholder_model,
    random_seeds={"model": RANDOM_SEED, "evaluation": RANDOM_SEED + 1},
)

bundle = build_temporal_dataset(engine=engine, spec=experiment_spec.dataset_spec)
split_result = FixedTemporalSplitter(experiment_spec.split).assign(bundle)

{
    "experiment_digest": experiment_spec.digest,
    "ticker_count": len(resolved_tickers),
    "generated_feature_count": len(bundle.manifest.feature_columns),
    "outer_train": split_result.summary("train").to_manifest(),
    "outer_validation": split_result.summary("validation").to_manifest(),
}


The canonical bundle is retained as the source of features, sample alignment, and the fixed outer
split. Its five-session target-end purge is conservative for the 1–4 session targets used below.
No test positions are requested anywhere in this notebook.

## 3. Construct aligned 1–5 session targets

In [ ]:
price_rows = load_bronze_daily_prices(
    engine=engine,
    provider=PROVIDER,
    tickers=resolved_tickers,
    end_date=DATA_CUTOFF,
    columns=("high", "low", "close", "adjusted_close"),
)
prices = price_rows.set_index(["provider", "ticker", "trading_date"]).sort_index()

if not bundle.features.index.isin(prices.index).all():
    raise ValueError(
        "The canonical feature bundle contains rows missing from loaded bronze prices."
    )


def build_short_horizon_targets(price_frame: pd.DataFrame) -> pd.DataFrame:
    """Build raw-return and ATR-unit targets in one split-adjusted price space."""
    numeric = price_frame.loc[:, ["high", "low", "close", "adjusted_close"]].apply(
        pd.to_numeric,
        errors="coerce",
    ).astype("float64")

    close = numeric["close"].mask(numeric["close"].le(0) | ~np.isfinite(numeric["close"]))
    adjusted_close = numeric["adjusted_close"].mask(
        numeric["adjusted_close"].le(0) | ~np.isfinite(numeric["adjusted_close"])
    )

    adjustment_factor = (adjusted_close / close).replace([np.inf, -np.inf], np.nan)
    adjusted_ohlc = pd.DataFrame(
        {
            "high": numeric["high"] * adjustment_factor,
            "low": numeric["low"] * adjustment_factor,
            "close": adjusted_close,
        },
        index=price_frame.index,
    )

    valid_bar = (
        adjusted_ohlc.notna().all(axis=1)
        & np.isfinite(adjusted_ohlc).all(axis=1)
        & adjusted_ohlc["high"].ge(adjusted_ohlc["low"])
        & adjusted_ohlc["high"].ge(adjusted_ohlc["close"])
        & adjusted_ohlc["low"].le(adjusted_ohlc["close"])
        & adjusted_ohlc["close"].gt(0)
    )
    adjusted_ohlc = adjusted_ohlc.where(valid_bar)

    forward = add_forward_return_targets(
        adjusted_ohlc.loc[:, ["close"]].rename(columns={"close": "adjusted_close"}),
        horizons=HORIZONS,
    )

    atr_values = atr(adjusted_ohlc, length=ATR_LENGTH).astype("float64")
    atr_fraction = atr_values / adjusted_ohlc["close"].abs()
    valid_atr = (
        atr_values.notna()
        & np.isfinite(atr_values)
        & atr_values.gt(0)
        & atr_fraction.ge(MIN_ATR_FRACTION_OF_PRICE)
    )
    effective_atr = atr_values.where(valid_atr).rename(f"effective_atr_{ATR_LENGTH}")

    grouped_adjusted_close = adjusted_ohlc["close"].groupby(
        level=["provider", "ticker"],
        sort=False,
    )

    result = pd.DataFrame(index=price_frame.index)
    result["adjustment_factor"] = adjustment_factor
    result["adjusted_high"] = adjusted_ohlc["high"]
    result["adjusted_low"] = adjusted_ohlc["low"]
    result["adjusted_close_for_targets"] = adjusted_ohlc["close"]
    result[f"atr_{ATR_LENGTH}"] = atr_values
    result[f"atr_fraction_{ATR_LENGTH}"] = atr_fraction
    result[f"effective_atr_{ATR_LENGTH}"] = effective_atr
    result["valid_adjusted_bar"] = valid_bar
    result["valid_atr"] = valid_atr

    for horizon in HORIZONS:
        result[f"raw_return_{horizon}d"] = forward[f"forward_return_{horizon}d"]

        future_adjusted_close = grouped_adjusted_close.shift(-horizon)
        valid = (
            adjusted_ohlc["close"].notna()
            & future_adjusted_close.notna()
            & effective_atr.notna()
            & np.isfinite(future_adjusted_close)
        )
        atr_units = (future_adjusted_close - adjusted_ohlc["close"]) / effective_atr
        result[f"atr_units_{horizon}d"] = atr_units.where(valid).astype("float64")

    return result


short_targets_full = build_short_horizon_targets(prices)
short_targets = short_targets_full.reindex(bundle.features.index)

raw_target_columns = [f"raw_return_{horizon}d" for horizon in HORIZONS]
atr_target_columns = [f"atr_units_{horizon}d" for horizon in HORIZONS]

short_targets.loc[:, raw_target_columns + atr_target_columns].describe(
    percentiles=[0.001, 0.01, 0.50, 0.99, 0.999]
).T

In [ ]:
target_quality_summary = {
    "source_rows": len(prices),
    "aligned_bundle_rows": len(short_targets),
    "invalid_adjusted_bar_fraction": float(
        (~short_targets["valid_adjusted_bar"].fillna(False)).mean()
    ),
    "invalid_atr_fraction": float(
        (~short_targets["valid_atr"].fillna(False)).mean()
    ),
    "atr_missing_fraction": float(short_targets[f"atr_{ATR_LENGTH}"].isna().mean()),
    "atr_fraction_distribution": (
        short_targets[f"atr_fraction_{ATR_LENGTH}"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .describe(percentiles=[0.001, 0.01, 0.50, 0.99, 0.999])
        .to_dict()
    ),
    "adjustment_factor_distribution": (
        short_targets["adjustment_factor"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .describe(percentiles=[0.001, 0.01, 0.50, 0.99, 0.999])
        .to_dict()
    ),
    "target_missing_fractions": (
        short_targets.loc[:, raw_target_columns + atr_target_columns]
        .isna()
        .mean()
        .to_dict()
    ),
}
target_quality_summary

In [ ]:
atr_outlier_rows = []
for column in atr_target_columns:
    extreme = short_targets[column].abs().gt(MAX_ABS_ATR_UNITS)
    if extreme.any():
        rows = short_targets.loc[
            extreme,
            [
                "adjustment_factor",
                "adjusted_high",
                "adjusted_low",
                "adjusted_close_for_targets",
                f"atr_{ATR_LENGTH}",
                f"atr_fraction_{ATR_LENGTH}",
                f"effective_atr_{ATR_LENGTH}",
                column,
            ],
        ].copy()
        rows.insert(0, "target_column", column)
        atr_outlier_rows.append(rows)

if atr_outlier_rows:
    atr_outliers = pd.concat(atr_outlier_rows).sort_values(
        atr_target_columns[0] if len(atr_target_columns) == 1 else "target_column"
    )
    display(atr_outliers.head(100))
    raise ValueError(
        "Implausible ATR-unit targets remain after adjustment and ATR validation. "
        "Inspect the displayed rows before fitting models."
    )

print(
    "ATR target validation passed: no absolute ATR-unit target exceeds "
    f"{MAX_ABS_ATR_UNITS:g}."
)

In [ ]:
atr_target_columns = [
    column
    for column in short_targets_full.columns
    if column.startswith("atr_units_")
]

diagnostic_columns = [
    "open",
    "high",
    "low",
    "close",
    "adjusted_close",
    "adjustment_factor",
    "adjusted_open",
    "adjusted_high",
    "adjusted_low",
    "adjusted_price_close",
    f"adjusted_atr_{ATR_LENGTH}",
    f"adjusted_atr_pct_{ATR_LENGTH}",
]

available_diagnostic_columns = [
    column
    for column in diagnostic_columns
    if column in price_frame.columns
]

target_diagnostics = price_frame[available_diagnostic_columns].copy()

outlier_frames = []

for target_column in atr_target_columns:
    target_values = short_targets_full[target_column]
    mask = target_values.abs().gt(MAX_ABS_ATR_UNITS)

    if not mask.any():
        continue

    outlier_index = target_values.index[mask]

    rows = target_diagnostics.reindex(outlier_index).copy()
    rows["target_column"] = target_column
    rows["target_value"] = target_values.loc[outlier_index]
    outlier_frames.append(rows)

if not outlier_frames:
    print("No ATR-unit outliers found.")
    atr_outliers = pd.DataFrame()
else:
    atr_outliers = (
        pd.concat(outlier_frames)
        .sort_values(
            "target_value",
            key=lambda values: values.abs(),
            ascending=False,
        )
    )

    display(atr_outliers.head(100))

### Target semantics

- `raw_return_hd` is the canonical adjusted-close forward return.
- `atr_units_hd` is the split-adjusted close change divided by split-adjusted ATR at the signal date.
- High, low, and close are transformed into the same adjusted price space before ATR is calculated.
- Zero, non-finite, or implausibly tiny ATR denominators are rejected rather than replaced with a microscopic floor.
- A fail-fast validation cell stops the experiment if any absolute ATR-unit target exceeds the configured plausibility bound.


## 4. Extract train and validation frames

In [ ]:
DATE_LEVEL = "trading_date"
TICKER_LEVEL = "ticker"

feature_columns = tuple(bundle.manifest.feature_columns)
train_positions = split_result.indices("train")
validation_positions = split_result.indices("validation")

X_all = bundle.features.loc[:, feature_columns]
metadata_all = bundle.samples


def target_column(target_family: str, horizon: int) -> str:
    if target_family not in TARGET_FAMILIES:
        raise KeyError(f"Unknown target family: {target_family!r}")
    return f"{target_family}_{horizon}d"


def make_experiment_frames(target_family: str, horizon: int):
    selected_target = target_column(target_family, horizon)
    raw_return_column = f"raw_return_{horizon}d"

    def build(positions):
        X = X_all.iloc[positions].copy()
        metadata = metadata_all.iloc[positions].copy()
        targets = short_targets.iloc[positions]

        y = pd.to_numeric(targets[selected_target], errors="coerce")
        actual_return = pd.to_numeric(targets[raw_return_column], errors="coerce")
        valid = y.notna() & actual_return.notna() & np.isfinite(y) & np.isfinite(actual_return)

        X = X.loc[valid]
        y = y.loc[valid].astype("float64")
        frame = metadata.loc[valid].copy()
        frame["actual_target"] = y
        frame["actual_return"] = actual_return.loc[valid].astype("float64")
        return X, y, frame

    return (*build(train_positions), *build(validation_positions))


shape_rows = []
for family in TARGET_FAMILIES:
    for horizon in HORIZONS:
        X_train, y_train, _, X_validation, y_validation, _ = make_experiment_frames(
            family,
            horizon,
        )
        shape_rows.append(
            {
                "target_family": family,
                "horizon": horizon,
                "train_rows": len(y_train),
                "validation_rows": len(y_validation),
                "feature_count": X_train.shape[1],
                "train_target_mean": y_train.mean(),
                "validation_target_mean": y_validation.mean(),
            }
        )

pd.DataFrame(shape_rows).set_index(["target_family", "horizon"])


## 5. Models and learning-to-rank labels

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRanker, XGBRegressor


def make_model(model_name: str):
    if model_name == "ridge":
        return Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
                ("regressor", Ridge(alpha=10.0)),
            ]
        )
    if model_name == "xgboost_regressor":
        return XGBRegressor(**XGB_REGRESSOR_PARAMS)
    if model_name == "xgboost_ranker":
        return XGBRanker(**XGB_RANKER_PARAMS)
    raise KeyError(f"Unknown model: {model_name!r}")


def date_values(frame: pd.DataFrame) -> pd.DatetimeIndex:
    if DATE_LEVEL in frame.index.names:
        return pd.DatetimeIndex(frame.index.get_level_values(DATE_LEVEL))
    if DATE_LEVEL in frame.columns:
        return pd.DatetimeIndex(frame[DATE_LEVEL])
    raise KeyError(f"{DATE_LEVEL!r} is unavailable in the frame.")


def ticker_values(frame: pd.DataFrame) -> np.ndarray:
    if TICKER_LEVEL in frame.index.names:
        return frame.index.get_level_values(TICKER_LEVEL).astype(str).to_numpy()
    if TICKER_LEVEL in frame.columns:
        return frame[TICKER_LEVEL].astype(str).to_numpy()
    return np.arange(len(frame)).astype(str)


def relevance_labels(target: pd.Series, metadata: pd.DataFrame) -> pd.Series:
    date_groups = pd.Series(
        date_values(metadata).to_numpy(),
        index=target.index,
        name="_date_group",
    )
    percentiles = target.groupby(date_groups, sort=False).rank(
        method="average",
        pct=True,
    )
    labels = np.searchsorted(
        np.asarray(RANK_RELEVANCE_QUANTILES, dtype="float64"),
        percentiles.to_numpy(dtype="float64"),
        side="right",
    )
    return pd.Series(labels.astype("int32"), index=target.index, name="relevance")


def ranking_training_inputs(
    X: pd.DataFrame,
    target: pd.Series,
    metadata: pd.DataFrame,
):
    dates = date_values(metadata)
    tickers = ticker_values(metadata)
    order = np.lexsort((tickers, dates.asi8))

    X_sorted = X.iloc[order]
    labels_sorted = relevance_labels(target, metadata).iloc[order]
    sorted_dates = dates[order]
    qid = pd.factorize(sorted_dates, sort=False)[0].astype("int32")

    if np.any(np.diff(qid) < 0):
        raise ValueError("Ranking query identifiers must be sorted in non-decreasing order.")
    return X_sorted, labels_sorted, qid


def fit_and_predict(
    model_name: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    train_metadata: pd.DataFrame,
    X_validation: pd.DataFrame,
):
    model = make_model(model_name)
    if model_name == "xgboost_ranker":
        X_rank, labels, qid = ranking_training_inputs(X_train, y_train, train_metadata)
        model.fit(X_rank, labels, qid=qid)
    else:
        model.fit(X_train, y_train)

    predictions = np.asarray(model.predict(X_validation), dtype="float64").reshape(-1)
    if len(predictions) != len(X_validation):
        raise ValueError("Model prediction count does not match validation rows.")
    return model, predictions


In [ ]:
# Inspect the graded relevance distribution before fitting any ranker.
_, example_target, example_metadata, _, _, _ = make_experiment_frames("raw_return", 5)
example_relevance = relevance_labels(example_target, example_metadata)
example_relevance.value_counts(normalize=True).sort_index().rename("fraction")


XGBRanker treats every signal date as one query group. Its labels are graded relevance levels,
not the continuous return itself. This lets the ranker optimize within-date ordering while all
models are still evaluated against the same continuous realized target and raw return.

## 6. Evaluation and tie-aware selection helpers

In [ ]:
from scipy.stats import kendalltau, pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score, r2_score


def group_by_date(frame: pd.DataFrame, *, sort: bool = True):
    if DATE_LEVEL in frame.index.names:
        return frame.groupby(level=DATE_LEVEL, sort=sort)
    if DATE_LEVEL in frame.columns:
        return frame.groupby(DATE_LEVEL, sort=sort)
    raise KeyError(f"{DATE_LEVEL!r} is unavailable in the frame.")


def finite_correlation(function, x, y) -> float:
    x_array = np.asarray(x, dtype="float64")
    y_array = np.asarray(y, dtype="float64")
    mask = np.isfinite(x_array) & np.isfinite(y_array)
    if mask.sum() < 3:
        return np.nan
    if np.ptp(x_array[mask]) == 0 or np.ptp(y_array[mask]) == 0:
        return np.nan
    result = function(x_array[mask], y_array[mask])
    return float(result.statistic if hasattr(result, "statistic") else result[0])


def add_predictions(frame: pd.DataFrame, predictions: np.ndarray) -> pd.DataFrame:
    prediction_array = np.asarray(predictions, dtype="float64").reshape(-1)
    if len(prediction_array) != len(frame):
        raise ValueError("Prediction count does not match frame rows.")

    result = frame.copy()
    result["predicted_score"] = prediction_array
    return result


def daily_ranking_metrics(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for signal_date, group in group_by_date(frame, sort=True):
        actual = group["actual_target"]
        predicted = group["predicted_score"]
        percentiles = actual.rank(method="average", pct=True)
        relevance = np.searchsorted(
            np.asarray(RANK_RELEVANCE_QUANTILES, dtype="float64"),
            percentiles.to_numpy(dtype="float64"),
            side="right",
        )
        ndcg = np.nan
        if len(group) >= 2 and np.max(relevance) > 0:
            ndcg = float(
                ndcg_score(
                    relevance.reshape(1, -1),
                    predicted.to_numpy(dtype="float64").reshape(1, -1),
                    k=min(5, len(group)),
                )
            )
        rows.append(
            {
                DATE_LEVEL: signal_date,
                "candidate_count": len(group),
                "spearman": finite_correlation(spearmanr, predicted, actual),
                "kendall_tau": finite_correlation(kendalltau, predicted, actual),
                "ndcg_at_5": ndcg,
            }
        )
    return pd.DataFrame(rows)


def prediction_cardinality_by_date(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for signal_date, group in group_by_date(frame, sort=True):
        scores = group["predicted_score"]
        unique_predictions = int(scores.nunique())
        maximum = scores.max()
        top_bucket_size = int(scores.eq(maximum).sum())
        rows.append(
            {
                DATE_LEVEL: signal_date,
                "candidate_count": len(group),
                "unique_predictions": unique_predictions,
                "top_bucket_size": top_bucket_size,
                "flat_score_date": unique_predictions == 1,
                "prediction_std": float(scores.std(ddof=0)),
            }
        )
    return pd.DataFrame(rows).set_index(DATE_LEVEL)


def aggregate_metrics(frame: pd.DataFrame, *, model_name: str) -> dict[str, float]:
    valid = np.isfinite(frame["actual_target"]) & np.isfinite(frame["predicted_score"])
    evaluated = frame.loc[valid]
    actual = evaluated["actual_target"].to_numpy(dtype="float64")
    predicted = evaluated["predicted_score"].to_numpy(dtype="float64")
    daily = daily_ranking_metrics(evaluated)
    cardinality = prediction_cardinality_by_date(evaluated)

    regression_metrics = {"mae": np.nan, "rmse": np.nan, "r2": np.nan}
    if model_name != "xgboost_ranker":
        regression_metrics = {
            "mae": mean_absolute_error(actual, predicted),
            "rmse": mean_squared_error(actual, predicted) ** 0.5,
            "r2": r2_score(actual, predicted),
        }

    return {
        "rows": len(evaluated),
        **regression_metrics,
        "pearson": finite_correlation(pearsonr, predicted, actual),
        "pooled_spearman": finite_correlation(spearmanr, predicted, actual),
        "pooled_kendall_tau": finite_correlation(kendalltau, predicted, actual),
        "mean_daily_spearman": daily["spearman"].mean(),
        "mean_daily_kendall_tau": daily["kendall_tau"].mean(),
        "mean_daily_ndcg_at_5": daily["ndcg_at_5"].mean(),
        "prediction_mean": predicted.mean(),
        "prediction_std": predicted.std(),
        "global_unique_predictions": np.unique(predicted).size,
        "median_daily_unique_predictions": cardinality["unique_predictions"].median(),
        "flat_score_date_fraction": cardinality["flat_score_date"].mean(),
        "median_top_bucket_size": cardinality["top_bucket_size"].median(),
    }


In [ ]:
def eligible_daily_group(group: pd.DataFrame) -> pd.DataFrame:
    valid = (
        np.isfinite(group["predicted_score"])
        & np.isfinite(group["actual_target"])
        & np.isfinite(group["actual_return"])
    )
    return group.loc[valid].copy()


def deterministic_order(group: pd.DataFrame) -> pd.DataFrame:
    ordered = group.copy()
    ordered["_ticker_sort"] = ticker_values(ordered)
    return ordered.sort_values(
        ["predicted_score", "_ticker_sort"],
        ascending=[False, True],
        kind="mergesort",
    ).drop(columns="_ticker_sort")


def select_daily_top(
    frame: pd.DataFrame,
    k: int,
    *,
    skip_flat_dates: bool = True,
) -> pd.DataFrame:
    selected = []
    for _, group in group_by_date(frame, sort=True):
        eligible = eligible_daily_group(group)
        if eligible.empty:
            continue
        if skip_flat_dates and eligible["predicted_score"].nunique() == 1:
            continue
        selected.append(deterministic_order(eligible).head(min(k, len(eligible))))

    if not selected:
        return frame.iloc[0:0].copy()
    return pd.concat(selected)


def select_daily_top_score_bucket(
    frame: pd.DataFrame,
    *,
    skip_flat_dates: bool = True,
) -> pd.DataFrame:
    selected = []
    for _, group in group_by_date(frame, sort=True):
        eligible = eligible_daily_group(group)
        if eligible.empty:
            continue
        if skip_flat_dates and eligible["predicted_score"].nunique() == 1:
            continue
        selected.append(eligible.loc[eligible["predicted_score"].eq(
            eligible["predicted_score"].max()
        )])

    if not selected:
        return frame.iloc[0:0].copy()
    return pd.concat(selected)


def selected_rows(frame: pd.DataFrame, selection: str) -> pd.DataFrame:
    if selection == "top_score_bucket":
        return select_daily_top_score_bucket(frame, skip_flat_dates=True)
    if selection.startswith("top_"):
        return select_daily_top(
            frame,
            int(selection.removeprefix("top_")),
            skip_flat_dates=True,
        )
    raise KeyError(f"Unknown selection: {selection!r}")


def daily_selection_outcomes(frame: pd.DataFrame, selection: str) -> pd.DataFrame:
    selected = selected_rows(frame, selection)
    if selected.empty:
        return pd.DataFrame(
            columns=[
                "selected_return",
                "selected_target",
                "selected_count",
                "universe_return",
                "return_spread",
            ]
        )

    selected_daily = group_by_date(selected, sort=True).agg(
        selected_return=("actual_return", "mean"),
        selected_target=("actual_target", "mean"),
        selected_count=("actual_return", "size"),
    )
    universe_daily = group_by_date(frame, sort=True)["actual_return"].mean()
    selected_daily["universe_return"] = universe_daily.reindex(selected_daily.index)
    selected_daily["return_spread"] = (
        selected_daily["selected_return"] - selected_daily["universe_return"]
    )
    return selected_daily


def selection_summary(frame: pd.DataFrame, selection: str) -> dict[str, float]:
    daily = daily_selection_outcomes(frame, selection)
    selected = selected_rows(frame, selection)
    return {
        "selection": selection,
        "selected_dates": len(daily),
        "selected_rows": len(selected),
        "mean_return": daily["selected_return"].mean(),
        "median_return": daily["selected_return"].median(),
        "positive_fraction": daily["selected_return"].gt(0).mean(),
        "mean_actual_target": daily["selected_target"].mean(),
        "mean_universe_return": daily["universe_return"].mean(),
        "mean_return_spread": daily["return_spread"].mean(),
        "median_return_spread": daily["return_spread"].median(),
        "mean_selected_count": daily["selected_count"].mean(),
    }


## 7. Fit the complete validation model matrix

In [ ]:
import gc
import time
from itertools import product

from tqdm.auto import tqdm

run_specs = list(product(TARGET_FAMILIES, HORIZONS, MODEL_NAMES))
metric_rows = []
prediction_frames = {}
fitted_models = {}

for family, horizon, model_name in tqdm(run_specs, desc="Short-horizon model matrix"):
    X_train, y_train, train_frame, X_validation, _, validation_frame = (
        make_experiment_frames(family, horizon)
    )

    started = time.perf_counter()
    try:
        model, predictions = fit_and_predict(
            model_name,
            X_train,
            y_train,
            train_frame,
            X_validation,
        )
    except Exception as error:
        raise RuntimeError(
            "Failed model run: "
            f"target_family={family}, horizon={horizon}, model={model_name}"
        ) from error

    elapsed_seconds = time.perf_counter() - started
    prediction_frame = add_predictions(validation_frame, predictions)
    key = (family, horizon, model_name)
    prediction_frames[key] = prediction_frame

    if STORE_FITTED_MODELS:
        fitted_models[key] = model

    metric_rows.append(
        {
            "target_family": family,
            "horizon": horizon,
            "model": model_name,
            "elapsed_seconds": elapsed_seconds,
            **aggregate_metrics(prediction_frame, model_name=model_name),
        }
    )

    if not STORE_FITTED_MODELS:
        del model
    gc.collect()

model_results = (
    pd.DataFrame(metric_rows)
    .set_index(["target_family", "horizon", "model"])
    .sort_index()
)
model_results


The locked test split has still not been requested. All model comparison and shortlisting below use
validation rows only.

## 8. Fixed top-k and complete top-score-bucket results

In [ ]:
selection_names = tuple(f"top_{k}" for k in TOP_K_VALUES) + ("top_score_bucket",)
selection_rows = []

for (family, horizon, model_name), frame in tqdm(
    prediction_frames.items(),
    desc="Tie-aware selection evaluation",
):
    for selection in selection_names:
        selection_rows.append(
            {
                "target_family": family,
                "horizon": horizon,
                "model": model_name,
                **selection_summary(frame, selection),
            }
        )

selection_results = (
    pd.DataFrame(selection_rows)
    .set_index(["target_family", "horizon", "model", "selection"])
    .sort_index()
)
selection_results


In [ ]:
top_1_comparison = (
    selection_results.xs("top_1", level="selection")
    .loc[:, [
        "selected_dates",
        "mean_return",
        "median_return",
        "positive_fraction",
        "mean_return_spread",
    ]]
    .sort_values("mean_return_spread", ascending=False)
)
top_1_comparison.head(15)


## 9. Validation subperiod stability

In [ ]:
subperiod_rows = []

for (family, horizon, model_name), frame in prediction_frames.items():
    for selection in selection_names:
        daily = daily_selection_outcomes(frame, selection)
        if daily.empty:
            continue
        dated = daily.copy()
        dated["year"] = pd.DatetimeIndex(dated.index).year
        for year, group in dated.groupby("year", sort=True):
            subperiod_rows.append(
                {
                    "target_family": family,
                    "horizon": horizon,
                    "model": model_name,
                    "selection": selection,
                    "year": int(year),
                    "dates": len(group),
                    "mean_return": group["selected_return"].mean(),
                    "median_return": group["selected_return"].median(),
                    "mean_return_spread": group["return_spread"].mean(),
                }
            )

subperiod_results = (
    pd.DataFrame(subperiod_rows)
    .set_index(["target_family", "horizon", "model", "selection", "year"])
    .sort_index()
)
subperiod_results


In [ ]:
subperiod_gate = (
    subperiod_results.reset_index()
    .groupby(["target_family", "horizon", "model", "selection"], sort=False)
    .agg(
        subperiod_count=("year", "nunique"),
        positive_spread_subperiods=(
            "mean_return_spread",
            lambda values: int(values.gt(0).sum()),
        ),
        minimum_subperiod_spread=("mean_return_spread", "min"),
        maximum_subperiod_spread=("mean_return_spread", "max"),
    )
)

candidate_table = selection_results.join(subperiod_gate)
candidate_table["all_subperiod_spreads_positive"] = (
    candidate_table["positive_spread_subperiods"]
    == candidate_table["subperiod_count"]
)

shortlist = candidate_table.loc[
    candidate_table["mean_return_spread"].gt(0)
    & candidate_table["median_return"].gt(0)
    & candidate_table["all_subperiod_spreads_positive"]
].sort_values(
    ["mean_return_spread", "positive_fraction"],
    ascending=[False, False],
).head(MAX_SHORTLIST_CONFIGURATIONS)

shortlist


### Decision gate

A configuration is provisionally shortlisted only when:

- its mean validation return spread is positive;

- its median selected return is positive; and

- its mean spread is positive in every validation year.

This gate is intentionally simple. Statistical significance is evaluated only for the highest-ranked
unique model/target/horizon configurations, and the locked test is still not opened.

## 10. Optional progress-visible bootstrap and permutation tests

In [ ]:
def bootstrap_selection_spreads(
    frame: pd.DataFrame,
    top_k_values=TOP_K_VALUES,
    *,
    iterations: int = BOOTSTRAP_ITERATIONS,
    seed: int = RANDOM_SEED,
    show_progress: bool = True,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []

    for k in tqdm(top_k_values, desc="Bootstrap top-k", disable=not show_progress):
        daily = daily_selection_outcomes(frame, f"top_{k}")
        spreads = daily["return_spread"].to_numpy(dtype="float64")
        estimates = np.empty(iterations, dtype="float64")
        for iteration in range(iterations):
            sampled = rng.choice(spreads, size=len(spreads), replace=True)
            estimates[iteration] = sampled.mean()

        rows.append(
            {
                "selection": f"top_{k}",
                "dates": len(spreads),
                "observed_mean_spread": spreads.mean(),
                "bootstrap_ci_low": np.quantile(estimates, 0.025),
                "bootstrap_ci_high": np.quantile(estimates, 0.975),
            }
        )
    return pd.DataFrame(rows).set_index("selection")


def permutation_groups(frame: pd.DataFrame):
    groups = []
    for _, group in group_by_date(frame, sort=True):
        eligible = eligible_daily_group(group)
        if eligible.empty or eligible["predicted_score"].nunique() == 1:
            continue
        groups.append(
            (
                eligible["predicted_score"].to_numpy(dtype="float64"),
                eligible["actual_return"].to_numpy(dtype="float64"),
                ticker_values(eligible),
            )
        )
    return groups


def within_date_permutation_tests(
    frame: pd.DataFrame,
    top_k_values=TOP_K_VALUES,
    *,
    iterations: int = PERMUTATION_ITERATIONS,
    seed: int = RANDOM_SEED,
    show_progress: bool = True,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    groups = permutation_groups(frame)
    if not groups:
        raise ValueError("No eligible non-flat dates remain for permutation testing.")

    observed = {
        k: daily_selection_outcomes(frame, f"top_{k}")["return_spread"].mean()
        for k in top_k_values
    }
    null_estimates = {
        k: np.empty(iterations, dtype="float64")
        for k in top_k_values
    }

    iterator = tqdm(
        range(iterations),
        desc="Within-date permutations",
        disable=not show_progress,
    )
    for iteration in iterator:
        daily_spreads = {k: [] for k in top_k_values}
        for scores, actual_returns, tickers in groups:
            permuted_scores = rng.permutation(scores)
            order = np.lexsort((tickers, -permuted_scores))
            universe_return = actual_returns.mean()
            for k in top_k_values:
                selected_return = actual_returns[order[: min(k, len(order))]].mean()
                daily_spreads[k].append(selected_return - universe_return)

        for k in top_k_values:
            null_estimates[k][iteration] = np.mean(daily_spreads[k])

    rows = []
    for k in top_k_values:
        null = null_estimates[k]
        observed_spread = observed[k]
        rows.append(
            {
                "selection": f"top_{k}",
                "observed_mean_spread": observed_spread,
                "null_mean_spread": null.mean(),
                "null_p95": np.quantile(null, 0.95),
                "one_sided_p_value": (
                    1 + np.sum(null >= observed_spread)
                ) / (iterations + 1),
            }
        )
    return pd.DataFrame(rows).set_index("selection")


In [ ]:
if RUN_RESAMPLING:
    unique_configurations = (
        shortlist.reset_index()
        .sort_values("mean_return_spread", ascending=False)
        .drop_duplicates(["target_family", "horizon", "model"])
        .head(RESAMPLING_CONFIGURATION_LIMIT)
    )

    if unique_configurations.empty:
        print("No configurations passed the validation shortlist gate; resampling was skipped.")
    else:
        resampling_rows = []
        for row in unique_configurations.itertuples(index=False):
            key = (row.target_family, row.horizon, row.model)
            frame = prediction_frames[key]
            bootstrap = bootstrap_selection_spreads(frame)
            permutation = within_date_permutation_tests(frame)
            combined = bootstrap.join(permutation, rsuffix="_permutation").reset_index()
            combined.insert(0, "model", row.model)
            combined.insert(0, "horizon", row.horizon)
            combined.insert(0, "target_family", row.target_family)
            resampling_rows.append(combined)

        resampling_results = pd.concat(resampling_rows, ignore_index=True)
        display(resampling_results)
else:
    print(
        "Resampling skipped. Review the shortlist first, then set RUN_RESAMPLING = True "
        "to test the leading unique configurations with visible progress."
    )


## 11. Compact comparison views

In [ ]:
import matplotlib.pyplot as plt

plot_data = (
    selection_results.xs("top_1", level="selection")["mean_return_spread"]
    .rename("mean_return_spread")
    .reset_index()
)

for family in TARGET_FAMILIES:
    family_data = plot_data.loc[plot_data["target_family"].eq(family)]
    pivot = family_data.pivot(index="horizon", columns="model", values="mean_return_spread")
    ax = pivot.plot(marker="o", figsize=(10, 5))
    ax.axhline(0, linewidth=1)
    ax.set_title(f"Top-1 mean return spread by horizon — {family}")
    ax.set_xlabel("Horizon (sessions)")
    ax.set_ylabel("Selected return minus same-date universe return")
    plt.show()


In [ ]:
shortlist_columns = [
    "selected_dates",
    "mean_return",
    "median_return",
    "positive_fraction",
    "mean_return_spread",
    "minimum_subperiod_spread",
    "median_top_bucket_size",
    "flat_score_date_fraction",
    "mean_daily_spearman",
    "mean_daily_ndcg_at_5",
]

model_metric_columns = [
    "median_top_bucket_size",
    "flat_score_date_fraction",
    "mean_daily_spearman",
    "mean_daily_ndcg_at_5",
]
shortlist_with_model_metrics = (
    shortlist.reset_index()
    .merge(
        model_results.loc[:, model_metric_columns].reset_index(),
        on=["target_family", "horizon", "model"],
        how="left",
        validate="many_to_one",
    )
    .set_index(["target_family", "horizon", "model", "selection"])
)
shortlist_with_model_metrics.loc[:, shortlist_columns]


## 12. Interpretation and handoff

Use the completed outputs to make one bounded decision:

- Advance at most two configurations if their top-k or complete top-score-bucket spread is
positive, stable in both validation years, and survives the matched permutation test.

- Retain Ridge only as a control , not as a candidate, unless it unexpectedly demonstrates
stable ranking value.

- Do not select a model from aggregate RMSE or (R^2) alone. The production decision is
cross-sectional selection, so daily ranking and selected-return spread are primary.

- Treat flat-score dates as no-trade dates. Never force a recommendation where the model has
no cross-sectional distinction.

- Treat a tied maximum as a candidate bucket. Evaluate the complete bucket rather than
pretending ticker-order tie-breaking is model information.

- Do not open the locked test during this notebook.

After this matrix is complete, the next notebook should be 04_excursion_and_barrier_analysis.ipynb . It should take no more than the two leading
configurations and test whether their predicted opportunity can be converted into attractive
MFE/MAE and stop-before-target behavior. Universe expansion and neural sequence models remain
separate later experiments.

## 13. Prediction-shape diagnostics

In [ ]:
ncols = 3
nsubplots = len(prediction_frames)
nrows = int(np.ceil(nsubplots / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(13, 2.6 * nrows),
    layout="constrained",
)
axes = np.asarray(axes).reshape(-1)

for ax, (key, frame) in zip(axes, prediction_frames.items()):
    ax.hexbin(
        x=frame["predicted_score"],
        y=frame["actual_return"],
        mincnt=1,
        bins="log",
        gridsize=200,
    )
    ax.grid()
    ax.set_title(" | ".join(str(part) for part in key), fontsize=11)
    ax.set_xlabel("Predicted score")
    ax.set_ylabel("Realized raw return")

for ax in axes[len(prediction_frames):]:
    ax.set_visible(False)

plt.show()

In [ ]:
prediction_shape_summary = (
    model_results.loc[
        :,
        [
            "prediction_mean",
            "prediction_std",
            "global_unique_predictions",
            "median_daily_unique_predictions",
            "flat_score_date_fraction",
            "median_top_bucket_size",
        ],
    ]
    .sort_index()
)
prediction_shape_summary